In [1]:
import pandas as pd

df = pd.read_parquet("../data/clean_4th_downs.parquet")


dfGO = df[df['decision'] == 'GO']
dfFieldGoal = df[df['decision'] == 'FIELD_GOAL']
dfPunt = df[df['decision'] == 'PUNT']

#Test for years 2021-2023, train on all others
df['season'].unique()

#Split the data now
dfGo_train = dfGO[dfGO['season'] <= 2021]
dfGo_test = dfGO[dfGO['season'] > 2021]
dfGo_train.shape
dfGo_test.shape

dfFieldGoal_train = dfFieldGoal[dfFieldGoal['season'] <= 2021]
dfFieldGoal_test = dfFieldGoal[dfFieldGoal['season'] > 2021]
dfFieldGoal_train.shape
dfFieldGoal_test.shape

dfPunt_train = dfPunt[dfPunt['season'] <= 2021]
dfPunt_test = dfPunt[dfPunt['season'] > 2021]
dfPunt_train.shape
dfPunt_test.shape

dfFieldGoal['yardline_100'].max()


np.float32(49.0)

In [2]:
dfGO

dfGO_train_features = dfGo_train[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfGO_train_features_scaled = (dfGO_train_features - dfGO_train_features.mean()) / dfGO_train_features.std()
dfGO_train_features_minMax = dfGO_train_features.agg(['min', 'max'])
dfGO_train_target = dfGo_train[['epa']]

dfGO_test_features = dfGo_test[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfGO_test_features_scaled = (dfGO_test_features - dfGO_train_features.mean()) / dfGO_train_features.std()
dfGO_test_target = dfGo_test[['epa']]

dfFieldGoal_train_features = dfFieldGoal_train[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfFieldGoal_train_features_scaled = (dfFieldGoal_train_features - dfFieldGoal_train_features.mean()) / dfFieldGoal_train_features.std()
dfFieldGoal_train_features_minMax = dfFieldGoal_train_features.agg(['min', 'max'])
dfFieldGoal_train_target = dfFieldGoal_train[['epa']]

dfFieldGoal_test_features = dfFieldGoal_test[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfFieldGoal_test_features_scaled = (dfFieldGoal_test_features - dfFieldGoal_train_features.mean()) / dfFieldGoal_train_features.std()
dfFieldGoal_test_target = dfFieldGoal_test[['epa']]

dfPunt_train_features = dfPunt_train[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfPunt_train_features_scaled = (dfPunt_train_features - dfPunt_train_features.mean()) / dfPunt_train_features.std()
dfPunt_train_features_minMax = dfPunt_train_features.agg(['min', 'max'])
dfPunt_train_target = dfPunt_train[['epa']]

dfPunt_test_features = dfPunt_test[['yardline_100', 'ydstogo', 'score_differential', 'game_seconds_remaining']]
dfPunt_test_features_scaled = (dfPunt_test_features - dfPunt_train_features.mean()) / dfPunt_train_features.std()
dfPunt_test_target = dfPunt_test[['epa']]



In [4]:
import torch

#X = feature, Y = target
def get_tensors(df_train_features, df_train_target):
    X_train = torch.tensor(df_train_features.values, dtype=torch.float32)
    Y_train = torch.tensor(df_train_target.values, dtype=torch.float32)
    return X_train, Y_train

X_train_GO, Y_train_GO = get_tensors(dfGO_train_features_scaled, dfGO_train_target)
X_train_FieldGoal, Y_train_FieldGoal = get_tensors(dfFieldGoal_train_features_scaled, dfFieldGoal_train_target)
X_train_Punt, Y_train_Punt = get_tensors(dfPunt_train_features_scaled, dfPunt_train_target)
X_test_GO, Y_test_GO = get_tensors(dfGO_test_features_scaled, dfGO_test_target)
X_test_FieldGoal, Y_test_FieldGoal = get_tensors(dfFieldGoal_test_features_scaled, dfFieldGoal_test_target)
X_test_Punt, Y_test_Punt = get_tensors(dfPunt_test_features_scaled, dfPunt_test_target)

X_train_Punt.shape


torch.Size([18747, 4])

In [5]:
import torch.nn as nn

#Set up the model for each decision type
model_GO = torch.nn.Sequential(
    torch.nn.Linear(4, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 1)
)

model_FieldGoal = torch.nn.Sequential(
    torch.nn.Linear(4, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 1)
)   

model_Punt = torch.nn.Sequential(
    torch.nn.Linear(4, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 1)
)

In [6]:
#Writing the training loop for each model
import torch.optim as optim

optimizerGO = optim.Adam(model_GO.parameters(), lr=0.001)
optimizerFieldGoal = optim.Adam(model_FieldGoal.parameters(), lr=0.001)
optimizerPunt = optim.Adam(model_Punt.parameters(), lr=0.001)

for epoch in range(1000):
    # Forward pass
    Y_pred_GO = model_GO(X_train_GO)
    Y_pred_FieldGoal = model_FieldGoal(X_train_FieldGoal)
    Y_pred_Punt = model_Punt(X_train_Punt)

    # Compute loss
    loss_GO = torch.mean((Y_pred_GO - Y_train_GO) ** 2)
    loss_FieldGoal = torch.mean((Y_pred_FieldGoal - Y_train_FieldGoal) ** 2)
    loss_Punt = torch.mean((Y_pred_Punt - Y_train_Punt) ** 2)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss GO: {loss_GO.item()}, Loss Field Goal: {loss_FieldGoal.item()}, Loss Punt: {loss_Punt.item()}")

    
    # Backward pass and optimization
    model_GO.zero_grad()
    model_FieldGoal.zero_grad()
    model_Punt.zero_grad()

    loss_GO.backward()
    loss_FieldGoal.backward()
    loss_Punt.backward()

    optimizerGO.step()
    optimizerFieldGoal.step()
    optimizerPunt.step()


Epoch 0: Loss GO: 9.00734806060791, Loss Field Goal: 2.470609664916992, Loss Punt: 1.1376738548278809
Epoch 100: Loss GO: 8.717870712280273, Loss Field Goal: 2.3929877281188965, Loss Punt: 1.0024577379226685
Epoch 200: Loss GO: 8.604888916015625, Loss Field Goal: 2.3867766857147217, Loss Punt: 0.9809923768043518
Epoch 300: Loss GO: 8.565048217773438, Loss Field Goal: 2.383934736251831, Loss Punt: 0.9709194302558899
Epoch 400: Loss GO: 8.542745590209961, Loss Field Goal: 2.3819613456726074, Loss Punt: 0.9661991596221924
Epoch 500: Loss GO: 8.530294418334961, Loss Field Goal: 2.3801751136779785, Loss Punt: 0.9637488126754761
Epoch 600: Loss GO: 8.519641876220703, Loss Field Goal: 2.379110813140869, Loss Punt: 0.9620017409324646
Epoch 700: Loss GO: 8.51347541809082, Loss Field Goal: 2.378593683242798, Loss Punt: 0.960546612739563
Epoch 800: Loss GO: 8.50866985321045, Loss Field Goal: 2.378359317779541, Loss Punt: 0.9588446617126465
Epoch 900: Loss GO: 8.502992630004883, Loss Field Goal: 2

In [7]:
#Checking to see if the model is working by predicting on the training data and comparing to the actual values  
predictionGO = model_GO(X_train_GO)
predictionGO = predictionGO.detach().numpy()
dfGo_train['predicted_epa'] = predictionGO
dfGo_train['ydstogo_bucket'] = pd.cut(dfGo_train['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfGo_train.groupby('ydstogo_bucket')['predicted_epa'].mean()

predictionFieldGoal = model_FieldGoal(X_train_FieldGoal)
predictionFieldGoal = predictionFieldGoal.detach().numpy()
dfFieldGoal_train['predicted_epa'] = predictionFieldGoal
dfFieldGoal_train['ydstogo_bucket'] = pd.cut(dfFieldGoal_train['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfFieldGoal_train.groupby('ydstogo_bucket')['predicted_epa'].mean()

predictionPunt = model_Punt(X_train_Punt)
predictionPunt = predictionPunt.detach().numpy()
dfPunt_train['predicted_epa'] = predictionPunt
dfPunt_train['ydstogo_bucket'] = pd.cut(dfPunt_train['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfPunt_train.groupby('ydstogo_bucket')['predicted_epa'].mean()


#Try on test data now
predictionGO_test = model_GO(X_test_GO)
predictionGO_test = predictionGO_test.detach().numpy()
dfGo_test['predicted_epa'] = predictionGO_test
dfGo_test['ydstogo_bucket'] = pd.cut(dfGo_test['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfGo_test.groupby('ydstogo_bucket')['predicted_epa'].mean()

predictionFieldGoal_test = model_FieldGoal(X_test_FieldGoal)
predictionFieldGoal_test = predictionFieldGoal_test.detach().numpy() 
dfFieldGoal_test['predicted_epa'] = predictionFieldGoal_test
dfFieldGoal_test['ydstogo_bucket'] = pd.cut(dfFieldGoal_test['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfFieldGoal_test.groupby('ydstogo_bucket')['predicted_epa'].mean()

predictionPunt_test = model_Punt(X_test_Punt)
predictionPunt_test = predictionPunt_test.detach().numpy()
dfPunt_test['predicted_epa'] = predictionPunt_test
dfPunt_test['ydstogo_bucket'] = pd.cut(dfPunt_test['ydstogo'], bins=[0, 2, 6, float('inf')], labels=['short', 'medium', 'long'])
dfPunt_test.groupby('ydstogo_bucket')['predicted_epa'].mean()       

ydstogo_bucket
short    -0.328124
medium   -0.185410
long     -0.020416
Name: predicted_epa, dtype: float32

In [8]:
#Save the models weights and biases for later use
torch.save(model_GO.state_dict(), '../backend/models/model_GO.pt')
torch.save(model_FieldGoal.state_dict(), '../backend/models/model_FieldGoal.pt')
torch.save(model_Punt.state_dict(), '../backend/models/model_Punt.pt')

In [3]:
goTrainMean = dfGO_train_features.mean()
goTrainstd = dfGO_train_features.std()
goTrainMin = dfGO_train_features_minMax.loc['min']
goTrainMax = dfGO_train_features_minMax.loc['max']

puntTrainMean = dfPunt_train_features.mean()
puntTrainstd = dfPunt_train_features.std()
puntTrainMin = dfPunt_train_features_minMax.loc['min']
puntTrainMax = dfPunt_train_features_minMax.loc['max']

fieldGoalTrainMean = dfFieldGoal_train_features.mean()
fieldGoalTrainstd = dfFieldGoal_train_features.std()
fieldGoalTrainMin = dfFieldGoal_train_features_minMax.loc['min']
fieldGoalTrainMax = dfFieldGoal_train_features_minMax.loc['max']

goTrainMeanDict = goTrainMean.to_dict()
goTrainstdDict = goTrainstd.to_dict()
goTrainMinDict = goTrainMin.to_dict()
goTrainMaxDict = goTrainMax.to_dict()

puntTrainMeanDict = puntTrainMean.to_dict()
puntTrainstdDict = puntTrainstd.to_dict()
puntTrainMinDict = puntTrainMin.to_dict()
puntTrainMaxDict = puntTrainMax.to_dict()

fieldGoalTrainMeanDict = fieldGoalTrainMean.to_dict()
fieldGoalTrainstdDict = fieldGoalTrainstd.to_dict()
fieldGoalTrainMinDict = fieldGoalTrainMin.to_dict()
fieldGoalTrainMaxDict = fieldGoalTrainMax.to_dict()

combinedDict = {'GO' : {'mean': goTrainMeanDict, 'std': goTrainstdDict, 'min' : goTrainMinDict, 'max' : goTrainMaxDict},
                'PUNT' : {'mean': puntTrainMeanDict, 'std': puntTrainstdDict, 'min' : puntTrainMinDict, 'max' : puntTrainMaxDict},
                'FIELD_GOAL' : {'mean': fieldGoalTrainMeanDict, 'std': fieldGoalTrainstdDict, 'min' : fieldGoalTrainMinDict, 'max' : fieldGoalTrainMaxDict}}





In [4]:
import json

with open('../backend/models/scaling_stats.json', 'w') as f:
    json.dump(combinedDict, f, indent=2)